In [ ]:
!pip -q install pypdf sentence-transformers openai scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 9.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import getpass
from pathlib import Path

import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from google.colab import files

In [ ]:
uploaded = files.upload()

pdf_paths = [Path(name) for name in uploaded.keys() if name.lower().endswith(".pdf")]

if not pdf_paths:
    raise ValueError("Please upload at least one PDF file.")

print("Uploaded papers:")
for path in pdf_paths:
    print("-", path.name)

Saving Transforming_Music_Recommendations_with_AI_Machine_Learning_amp_Deep_Learning_Solutions.pdf to Transforming_Music_Recommendations_with_AI_Machine_Learning_amp_Deep_Learning_Solutions.pdf
Uploaded papers:
- Transforming_Music_Recommendations_with_AI_Machine_Learning_amp_Deep_Learning_Solutions.pdf


In [ ]:
def extract_pdf_pages(pdf_path):
    """Return one record per PDF page."""
    reader = PdfReader(str(pdf_path))
    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = re.sub(r"\s+", " ", text).strip()

        if text:
            pages.append({
                "paper": pdf_path.name,
                "page": page_number,
                "text": text
            })

    return pages

pages = []

for pdf_path in pdf_paths:
    extracted_pages = extract_pdf_pages(pdf_path)
    pages.extend(extracted_pages)
    print(f"{pdf_path.name}: extracted {len(extracted_pages)} text pages")

if not pages:
    raise ValueError(
        "No readable text was extracted. The PDF may be scanned and require OCR."
    )

Transforming_Music_Recommendations_with_AI_Machine_Learning_amp_Deep_Learning_Solutions.pdf: extracted 5 text pages


In [ ]:
CHUNK_SIZE = 220       # Approximate words per chunk
CHUNK_OVERLAP = 40     # Shared words between consecutive chunks

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if len(chunk.strip()) > 50:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

chunks = []

for page in pages:
    page_chunks = chunk_text(page["text"])

    for index, chunk in enumerate(page_chunks, start=1):
        chunks.append({
            "id": len(chunks) + 1,
            "paper": page["paper"],
            "page": page["page"],
            "chunk_number": index,
            "text": chunk
        })

print(f"Created {len(chunks)} chunks from {len(pages)} pages.")
print("\nExample chunk:\n")
print(chunks[0]["text"][:700])

Created 23 chunks from 5 pages.

Example chunk:

Transforming Music Recommendations with AI: Machine Learning & Deep Learning Solutions 1L Aslesha Chilakamarri Dept. of Computer Science and Engineering, Vignan’s Institute of Engineering for Women(A), Vishakapatnam, India chilakamarriaslesha@gmail.com 4Sai Siri Emandi Dept. of Computer Science of Engineering, Vignan’s Institute of Engineering for Women(A) Vishakapatnam, India saisiriemandi117@gmail.com 2Yaswanthi Devi Notla Dept. of Computer Science of Engineering, Vignan’s Institute of Engineering for Women(A), Vishakapatnam, India n.yaswanthi2004@gmail.com 5Ramya Sree Salapu Dept. of Electrial and Electronics Engineering, Vignan’s Institute of Engineering for Women(A) Vishakapatnam, India


In [ ]:
# A compact, free local embedding model.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [chunk["text"] for chunk in chunks]

# normalize_embeddings=True makes dot product equivalent to cosine similarity.
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
TOP_K = 5  # Number of most relevant chunks supplied to the LLM

def retrieve_relevant_chunks(question, top_k=TOP_K):
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    similarities = chunk_embeddings @ question_embedding
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for rank, index in enumerate(top_indices, start=1):
        item = chunks[index].copy()
        item["score"] = float(similarities[index])
        item["source_id"] = f"S{rank}"
        results.append(item)

    return results

In [ ]:
!pip -q install google-genai

from google import genai
from google.genai import types

GEMINI_API_KEY = getpass.getpass("Enter your Google Gemini API key: ")

client = genai.Client(api_key=GEMINI_API_KEY)

# You can also try: "gemini-flash-latest"
MODEL = "gemini-2.5-flash"

Enter your Google Gemini API key: ··········


In [ ]:
def answer_question(question, top_k=TOP_K):
    retrieved_chunks = retrieve_relevant_chunks(question, top_k=top_k)

    context_blocks = []

    for item in retrieved_chunks:
        context_blocks.append(
            f"[{item['source_id']}] "
            f"Paper: {item['paper']} | Page: {item['page']} | "
            f"Similarity: {item['score']:.3f}\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_blocks)

    system_instruction = """
You are a research-paper question-answering assistant.

Answer using ONLY the supplied research-paper context.
Do not use outside knowledge.

If the answer is not supported by the context, say:
"I could not find this information in the uploaded papers."

Rules:
1. Give a direct and clear answer.
2. Cite every factual claim using source labels such as [S1].
3. Do not invent methods, datasets, findings, limitations, or citations.
4. If sources disagree, explain the disagreement and cite both sources.
5. End with a short Sources section listing cited source labels,
   paper name, and page number.
"""

    prompt = f"""
Question:
{question}

Retrieved research-paper context:
{context}
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.2,
            max_output_tokens=1000
        )
    )

    answer = response.text

    if not answer:
        answer = "The model did not return an answer. Please try again."

    return answer, retrieved_chunks

In [ ]:
MODEL = "gemini-3.6-flash"
question = "What is the objective of the paper?"

answer, retrieved = answer_question(question)

print(answer)

The objective of the paper is to provide users with personalized, on-demand music recommendations tailored to their real-time emotional state [S5]. The system utilizes a standard webcam to capture video, deep learning and visual concept recognition to detect the user's mood, and an automated YouTube search to deliver matching songs without requiring manual input or wearable devices [S5]. Ultimately, the project aims to bridge the gap between human emotion and music recommendation technology, creating a more empathetic, responsive, and adaptive interaction [S5].

### Sources
- **[S5]** Paper: *Transforming_Music_Recommendations_with_AI_Machine_Learning_amp_Deep_Learning_Solutions.pdf*, Page: 2


In [ ]:
for item in retrieved:
    print(
        f"\n[{item['source_id']}] {item['paper']} — "
        f"Page {item['page']} "
        f"(Similarity: {item['score']:.3f})"
    )
    print(item["text"][:600], "...")


[S1] Transforming_Music_Recommendations_with_AI_Machine_Learning_amp_Deep_Learning_Solutions.pdf — Page 5 (Similarity: 0.217)
Music Recommendatio n System." arXiv preprint arXiv:2503.20739 (2025), DOI: 10.48550/arXiv.2503.20739. [7] Liu, Yingchia, Yang Xu, and Shiji Zhou. "Enhancing user experience through machine learning-based personalized recomme ndation systems: Behavior data-driven UI design." Authorea Preprints (2024). [8] Livingstone, Steven R., Ralf Mühlberger, Andrew R. Brown, and Andrew Loch. "Controlling musical emotionality: An affective computational architecture for influencing musical emotions." Digital Creativity 18, no. 1 (2007): 43-53, DOI: 10.1080/14626260701253606. [9] Mizgajski, Jan, and Mikołaj  ...

[S2] Transforming_Music_Recommendations_with_AI_Machine_Learning_amp_Deep_Learning_Solutions.pdf — Page 5 (Similarity: 0.175)
desired emotions." In Proceedings of the 17th international conference on mobile and ubiquitous m ultimedia , pp. 205-213, November 2018, DOI

In [ ]:
while True:
    question = input("\nAsk a question about the uploaded papers (or type 'exit'): ")

    if question.lower().strip() in {"exit", "quit"}:
        print("Session ended.")
        break

    answer, _ = answer_question(question)

    print("\nAnswer:\n")
    print(answer)


Ask a question about the uploaded papers (or type 'exit'): what is deep learning

Answer:

Based on the provided papers, a formal general definition of "deep learning" is not explicitly provided. 

However, the context describes deep learning in terms of deep neural network models—such
